# 81 Candidates, judge scores and labels (A100)

For validation (579 questions), dev (1,534) and mini-dev (500):

1. **Candidates** (vLLM, about 15 minutes): the generator's greedy answer (already predicted) plus 8 samples with its normal prompt and 8 with the "efficient" prompt, at temperature 1. Duplicates are merged, so a question has up to 17 distinct candidates.
2. **Judge scores** (about 20 minutes): p (probability correct) and c (relative cost) for every candidate. Nothing is executed.
3. **Labels** (CPU of this machine, about 25 minutes): every candidate is executed once to know whether it is correct and what it costs. These labels are used only to choose the shortlist margin on validation and for the analysis, never to answer.

About **1 hour, about 12 units**. Every step is resumable: after a disconnect, run the setup, install and config cells again, then the cell that was running.

**After every code update, reopen this notebook from the GitHub link.**

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = True  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
# Training libraries (transformers and peft, pinned in pyproject.toml); vLLM above brings torch.
!pip install -q -e "{REPO_DIR}[train]"
# Colab ships torchao 0.10.0, which peft rejects with an error (it needs 0.16 or newer). We do not use it.
!pip uninstall -q -y torchao
import importlib.metadata as md
print({p: md.version(p) for p in ("torch", "transformers", "peft", "vllm")})

In [ ]:
import json, os
R = f"{DRIVE_ROOT}/effsql/runs"
DATA = "judge_data_v1"    # the judge's training data (built from own_v1 and pool_v1, already on Drive)
JUDGE = "judge_v1"        # the judge: adapter + cost head
CAND = "judge_cand_v1"    # candidates, scores and labels for val, dev and mini-dev
SEL = "judge_v1_sel"      # the selected answers (also judge_v1_sel_p and judge_v1_sel_vote)
GEN = "rl_v1_it8"         # the generator
VAL_DBS = ",".join(json.load(open(f"{R}/sft_v1/export_meta.json"))["holdout_db_ids"])
assert json.load(open(f"{R}/{JUDGE}/judge_meta.json"))["complete"], "run 80_judge_train first"
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("CPU cores:", os.cpu_count())

### 1. Candidates (vLLM)

In [ ]:
!python -m effsql.judge candidates --run {CAND} --splits val,dev,minidev

### 2. Judge scores (p and c per candidate)

In [ ]:
!python -m effsql.judge score --run {CAND} --judge {JUDGE} --splits val,dev,minidev

### 3. Stage the databases, then execute every candidate (labels)

In [ ]:
!effsql stage --split train --db-ids {VAL_DBS}
!effsql stage --split dev
!python -m effsql.judge label --run {CAND} --splits val,dev,minidev

### Done: disconnect now, then run `82_judge_report` on a CPU runtime

*Runtime > Disconnect and delete runtime.*